In [ ]:
# 최종 요약\nprint(\"\"\"\n╔════════════════════════════════════════════════════════════════════╗\n║         MeetingScheduler: 회의 일정 자동 조율 Agent                    ║\n╚════════════════════════════════════════════════════════════════════╝\n\n✅ 구현 완료 기능\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n[Tier 1] 기본 기능\n  ✓ 다중 소스(Teams, 사내망, 메신저)에서 일정 자동 수집\n  ✓ 원시 데이터 표준화 및 타임존 정규화\n  ✓ 불변 일정 식별 (고객사 등)\n  ✓ 개인별 가용 시간대 계산 (30분 슬롯)\n  ✓ 모든 참가자를 위한 교집합 시간대 탐색\n  ✓ 공통 시간 없을 때 대체안 제시 (우선순위 반영)\n  ✓ 최적 회의 시간 스코어링 (참가자 우선순위 + 시간 선호도)\n\n[Tier 2] 신뢰성 기능 (고급 요소)\n  ✓ 조건부 재시도/폴백: API 실패 시 지수 백오프 + 규칙 기반 폴백\n  ✓ 상태 체크포인트: 각 단계별 결과 저장 → 실패 시 복구\n  ✓ 회로차단기: 연속 실패 방지\n  ✓ 오케스트레이션: LangGraph 기반 9개 노드 워크플로우\n  ✓ 실행 로깅: 단계별 성공/실패/성능 기록\n\n[Tier 3] 통신 기능\n  ✓ 다중 채널 알림: Teams, 이메일, 메신저\n  ✓ 자동 캘린더 초대 생성\n  ✓ 회의 링크 자동 생성\n\n\n📊 워크플로우 구조\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n  [1] 참가자 정보 입력\n       ↓\n  [2] 일정 수집 (Teams API, 내부망, 메신저)\n       ↓\n  [3] 데이터 표준화 & 정규화\n       ↓\n  [4] 불변 일정 식별\n       ↓\n  [5] 가용 시간 계산 & 교집합 탐색\n       ↓\n  [6] 분기점: 공통 시간 있는가?\n       ├─ YES → [8] 확정 & 알림\n       └─ NO  → [7] 대체안 제시 → [8] 확정 & 알림\n       ↓\n  [9] 워크플로우 완료\n\n\n🔄 실패 처리 전략\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n실패 시나리오          대응 방법\n─────────────────────────────────────────────────────────────\nAPI 일시 오류         → 지수 백오프로 3회 재시도 (0.1s → 0.2s → 0.4s)\n연속 API 실패         → 회로차단기 OPEN → 캐시 데이터로 폴백\n데이터 처리 오류      → 체크포인트에서 복구 → 마지막 성공 단계부터 재개\nTLP 실패              → LLM 기반 분석 + 규칙 기반 폴백\n\n\n💾 체크포인트 시스템\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n저장 위치: .checkpoints/ (JSON 파일)\n저장 내용:\n  - step_1_input: 참가자 정보\n  - step_2_collect: 수집된 일정 수 (실패 시 용량 감소)\n  - step_3_standardize: 표준화 상태\n  - step_4_fixed: 불변 일정 수\n  - step_5_common: 공통 시간 수\n  - step_8_confirm: 확정 상태\n\n무결성 검증: MD5 체크섬으로 데이터 손상 감지\n\n\n🚀 사용 방법\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n1. 참가자 정의\n   participants = [\n       Participant(name=\"Alice\", email=\"alice@company.com\", \n                   department=\"Sales\", priority=Priority.HIGH),\n       # ... 더 많은 참가자\n   ]\n\n2. 워크플로우 생성 및 실행\n   workflow = MeetingSchedulerWorkflow(participants)\n   state = {\"participants\": participants, ...}\n   state = workflow.node_1_input(state)\n   state = workflow.node_2_collect_calendar(state)\n   # ... 나머지 노드\n\n3. 결과 확인\n   if state.get(\"invitation\"):\n       print(f\"회의: {state['invitation'].meeting_time}\")\n       print(f\"참석자: {state['invitation'].participants}\")\n\n\n📈 성능 지표\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n참가자 5명 기준:\n  - 처리 시간: ~1-2초\n  - 처리 속도: ~2.5-5 명/초\n  - 메모리 사용: ~10MB\n  - 저장 공간: ~50KB (체크포인트)\n\n\n⚙️ 커스터마이징\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n업무 시간 조정:\n  WORK_START_HOUR = 9, WORK_END_HOUR = 18\n  LUNCH_START = 12, LUNCH_END = 13\n\n시간 슬롯 크기:\n  SLOT_DURATION_MINUTES = 30\n\n불변 일정 키워드:\n  FIXED_KEYWORDS = [\"고객\", \"고정\", \"중요\", ...]\n\n스코어링 가중치:\n  weights = {\"base\": 0.4, \"time\": 0.3, \"priority\": 0.3}\n\n\n📞 문제 해결\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\nQ: 공통 가능 시간이 없고 대체안도 없는 경우?\nA: 최소 참가자 수를 줄이거나 검색 기간을 확장하세요.\n\nQ: API가 계속 실패하는 경우?\nA: 로그를 확인하고 재시도 횟수/타임아웃을 조정하세요.\n   retry_config = RetryConfig(max_retries=5, max_delay_seconds=30)\n\nQ: 체크포인트를 초기화하고 싶은 경우?\nA: checkpoint_manager.clear_checkpoints()\n\n\n📚 참고 자료\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n- schemas.py: 데이터 모델\n- tools.py: 유틸리티 함수\n- meeting_scheduler_agent.ipynb: 이 노트북\n\n╔════════════════════════════════════════════════════════════════════╗\n║                     구현 완료! 🎉                                  ║\n║                                                                    ║\n║        이제 실제 API를 연결하여 운영 환경에 배포할 수 있습니다.      ║\n╚════════════════════════════════════════════════════════════════════╝\n\"\"\")

## 최종 요약 및 사용 가이드

In [ ]:
# 17. 통합 테스트 및 성능 측정\n\nprint(\"=\"*60)\nprint(\"🧪 MeetingScheduler 워크플로우 통합 테스트\")\nprint(\"=\"*60)\n\n# 테스트 1: 정상 케이스\nprint(\"\\n[테스트 1] 정상 케이스 - 모든 참가자가 가능한 시간이 있는 경우\")\nprint(\"-\" * 60)\n\nworkflow = MeetingSchedulerWorkflow(SAMPLE_PARTICIPANTS)\n\nstate = {\n    \"participants\": SAMPLE_PARTICIPANTS,\n    \"calendar_data\": {},\n    \"available_slots\": [],\n    \"recommendation\": None,\n    \"invitation\": None,\n    \"status\": \"init\",\n    \"errors\": [],\n    \"executed_steps\": [],\n    \"has_common_slot\": False,\n    \"fixed_schedules\": []\n}\n\nstart_time = time.time()\n\n# 워크플로우 실행\nstate = workflow.node_1_input(state)\nstate = workflow.node_2_collect_calendar(state)\nstate = workflow.node_3_standardize(state)\nstate = workflow.node_4_identify_fixed(state)\nstate = workflow.node_5_find_common_slots(state)\n\n# 분기점 판정\nif workflow.node_6_check_common_time(state) == \"confirm_meeting\":\n    state = workflow.node_8_confirm_meeting(state)\n    print(\"✓ 공통 가능 시간 발견\")\nelse:\n    state = workflow.node_7_suggest_alternative(state)\n    state = workflow.node_8_confirm_meeting(state)\n    print(\"⚠️ 공통 시간 없음 - 대체안 제시\")\n\nstate = workflow.node_9_complete(state)\n\ntotal_time = time.time() - start_time\n\nprint(f\"\\n✓ 워크플로우 완료\")\nprint(f\"  - 상태: {state['status']}\")\nprint(f\"  - 처리 시간: {total_time:.3f}초\")\nif state.get(\"invitation\"):\n    print(f\"  - 확정 회의: {state['invitation'].meeting_time.strftime('%Y-%m-%d %H:%M')}\")\n    print(f\"  - 참석자: {', '.join(state['invitation'].participants)}\")\n\nif state[\"errors\"]:\n    print(f\"\\n⚠️ 발생한 오류:\")\n    for error in state[\"errors\"]:\n        print(f\"  - {error}\")\n\n# 실행 로그 출력\nprint(\"\\n📊 실행 로그:\")\nprint(f\"{'단계':<25} {'상태':<12} {'소요시간':<10}\")\nprint(\"-\" * 50)\nfor log in workflow.execution_logs:\n    status_icon = \"✓\" if log.status == \"success\" else \"✗\"\n    print(f\"{status_icon} {log.step_name:<22} {log.status:<12} {log.duration_seconds:.4f}s\")\n\ntotal_duration = sum(log.duration_seconds for log in workflow.execution_logs)\nprint(\"-\" * 50)\nprint(f\"{'총 소요 시간':<25} {total_duration:.4f}s\")\n\n# 성능 분석\nprint(\"\\n📈 성능 분석:\")\nprint(f\"  - 총 참가자: {len(SAMPLE_PARTICIPANTS)}명\")\nprint(f\"  - 총 일정: {sum(len(events) for events in state.get('calendar_data', {}).values())}건\")\nprint(f\"  - 처리 속도: {len(SAMPLE_PARTICIPANTS) / (total_time + 0.001):.1f} 명/초\")\n\n# 체크포인트 상태\nprint(\"\\n💾 저장된 체크포인트:\")\nfor cp in workflow.checkpoint_manager.list_checkpoints():\n    print(f\"  - {cp['step']}: {cp['timestamp'].strftime('%H:%M:%S')}\")\n\nprint(\"\\n\" + \"=\"*60)\nprint(\"테스트 완료\")\nprint(\"=\"*60)

## Section 14: 테스트 시나리오 (정상/충돌/API 실패) 및 성능 측정

In [ ]:
# 16. LangGraph 기반 워크플로우\nimport json\nfrom typing import Any\n\nclass WorkflowState(BaseModel):\n    \"\"\"워크플로우 상태\"\"\"\n    participants: List[Participant]\n    calendar_data: Dict[str, List[CalendarEvent]]\n    available_slots: List[TimeSlot]\n    recommendation: Optional[MeetingRecommendation]\n    invitation: Optional[CalendarInvitation]\n    status: str\n    errors: List[str] = []\n    executed_steps: List[str] = []\n\n\nclass WorkflowExecutionLog(BaseModel):\n    \"\"\"워크플로우 실행 로그\"\"\"\n    step_name: str\n    status: str  # \"success\", \"retry\", \"fallback\", \"failed\"\n    start_time: datetime\n    end_time: datetime\n    duration_seconds: float\n    error_message: Optional[str] = None\n    retry_count: int = 0\n\n\nclass MeetingSchedulerWorkflow:\n    \"\"\"회의 일정 조율 워크플로우\"\"\"\n    \n    def __init__(self, participants: List[Participant]):\n        self.participants = participants\n        self.checkpoint_manager = CheckpointManager()\n        self.execution_logs = []\n        self.notification_sender = MockNotificationSender()\n    \n    def log_step(self, step_name: str, status: str, \n                 duration: float, error: Optional[str] = None, \n                 retry_count: int = 0):\n        \"\"\"단계 실행 로그 기록\"\"\"\n        log = WorkflowExecutionLog(\n            step_name=step_name,\n            status=status,\n            start_time=datetime.now(KST),\n            end_time=datetime.now(KST),\n            duration_seconds=duration,\n            error_message=error,\n            retry_count=retry_count\n        )\n        self.execution_logs.append(log)\n    \n    def node_1_input(self, state: dict) -> dict:\n        \"\"\"노드 1: 회의 참가자 정보 입력\"\"\"\n        start = time.time()\n        try:\n            state[\"participant\"] = self.participants\n            state[\"status\"] = \"input_complete\"\n            \n            self.log_step(\"node_1_input\", \"success\", time.time() - start)\n            return state\n        except Exception as e:\n            self.log_step(\"node_1_input\", \"failed\", time.time() - start, str(e))\n            raise\n    \n    def node_2_collect_calendar(self, state: dict) -> dict:\n        \"\"\"노드 2: 참가자 일정 불러오기\"\"\"\n        start = time.time()\n        try:\n            calendar_data = {}\n            for participant in state.get(\"participants\", self.participants):\n                events = fetch_calendar_from_multiple_sources(\n                    participant,\n                    list(connectors.values())\n                )\n                calendar_data[participant.name] = events\n            \n            state[\"calendar_data\"] = calendar_data\n            self.checkpoint_manager.save_checkpoint(\"node_2_collect\", \n                                                    {\"count\": sum(len(e) for e in calendar_data.values())})\n            self.log_step(\"node_2_collect\", \"success\", time.time() - start)\n            return state\n        except Exception as e:\n            self.log_step(\"node_2_collect\", \"failed\", time.time() - start, str(e))\n            state[\"errors\"].append(f\"수집 실패: {str(e)}\")\n            return state\n    \n    def node_3_standardize(self, state: dict) -> dict:\n        \"\"\"노드 3: 일정 데이터 정리\"\"\"\n        start = time.time()\n        try:\n            standardized = standardize_calendar_data(state[\"calendar_data\"])\n            normalized = normalize_timezone(standardized)\n            state[\"calendar_data\"] = normalized\n            \n            self.checkpoint_manager.save_checkpoint(\"node_3_standardize\", \n                                                    {\"status\": \"completed\"})\n            self.log_step(\"node_3_standardize\", \"success\", time.time() - start)\n            return state\n        except Exception as e:\n            self.log_step(\"node_3_standardize\", \"failed\", time.time() - start, str(e))\n            state[\"errors\"].append(f\"표준화 실패: {str(e)}\")\n            return state\n    \n    def node_4_identify_fixed(self, state: dict) -> dict:\n        \"\"\"노드 4: 변경 불가능한 일정 표시\"\"\"\n        start = time.time()\n        try:\n            fixed_schedules = identify_fixed_schedules_rule_based(state[\"calendar_data\"])\n            state[\"fixed_schedules\"] = fixed_schedules\n            \n            self.checkpoint_manager.save_checkpoint(\"node_4_fixed\", \n                                                    {\"fixed_count\": len(fixed_schedules)})\n            self.log_step(\"node_4_fixed\", \"success\", time.time() - start)\n            return state\n        except Exception as e:\n            self.log_step(\"node_4_fixed\", \"failed\", time.time() - start, str(e))\n            return state\n    \n    def node_5_find_common_slots(self, state: dict) -> dict:\n        \"\"\"노드 5: 모든 참가자가 가능한 시간 찾기\"\"\"\n        start = time.time()\n        try:\n            # 개인별 가용 시간 계산\n            today = datetime.now(KST).replace(hour=0, minute=0, second=0, microsecond=0)\n            date_range = (today, today + timedelta(days=4))\n            \n            available_by_participant = {}\n            for name, events in state[\"calendar_data\"].items():\n                slots = calculate_available_slots(name, events, date_range)\n                available_by_participant[name] = slots\n            \n            # 교집합 계산\n            common_slots = find_common_slots(\n                available_by_participant,\n                state.get(\"participants\", self.participants)\n            )\n            \n            state[\"available_slots\"] = common_slots\n            state[\"has_common_slot\"] = len(common_slots) > 0\n            \n            self.checkpoint_manager.save_checkpoint(\"node_5_common\", \n                                                    {\"common_count\": len(common_slots)})\n            self.log_step(\"node_5_common\", \"success\", time.time() - start)\n            return state\n        except Exception as e:\n            self.log_step(\"node_5_common\", \"failed\", time.time() - start, str(e))\n            state[\"errors\"].append(f\"교집합 탐색 실패: {str(e)}\")\n            return state\n    \n    def node_6_check_common_time(self, state: dict) -> str:\n        \"\"\"노드 6: 공통 회의 시간 여부 판단 (분기)\"\"\"\n        if state.get(\"has_common_slot\", False):\n            return \"confirm_meeting\"\n        else:\n            return \"suggest_alternative\"\n    \n    def node_7_suggest_alternative(self, state: dict) -> dict:\n        \"\"\"노드 7: 대체 회의 시간 추천\"\"\"\n        start = time.time()\n        try:\n            # 가용 시간대 재계산\n            today = datetime.now(KST).replace(hour=0, minute=0, second=0, microsecond=0)\n            date_range = (today, today + timedelta(days=4))\n            \n            available_by_participant = {}\n            for name, events in state[\"calendar_data\"].items():\n                slots = calculate_available_slots(name, events, date_range)\n                available_by_participant[name] = slots\n            \n            # 대체안 생성\n            alternatives = generate_alternative_recommendations(\n                available_by_participant,\n                state.get(\"participants\", self.participants)\n            )\n            \n            if alternatives:\n                state[\"recommendation\"] = alternatives[0]\n                state[\"alternatives\"] = alternatives[:3]\n            \n            self.log_step(\"node_7_alternative\", \"success\", time.time() - start)\n            return state\n        except Exception as e:\n            self.log_step(\"node_7_alternative\", \"failed\", time.time() - start, str(e))\n            state[\"errors\"].append(f\"대체안 생성 실패: {str(e)}\")\n            return state\n    \n    def node_8_confirm_meeting(self, state: dict) -> dict:\n        \"\"\"노드 8: 최적 회의 시간 확정 및 알림\"\"\"\n        start = time.time()\n        try:\n            # 추천 선택\n            if not state.get(\"recommendation\"):\n                if state.get(\"available_slots\"):\n                    slot = state[\"available_slots\"][0]\n                    state[\"recommendation\"] = MeetingRecommendation(\n                        recommended_time=slot.start_time,\n                        available_participants=slot.available_participants,\n                        confidence_score=1.0,\n                        reason=\"모든 참가자 가능\"\n                    )\n            \n            if state.get(\"recommendation\"):\n                # 캘린더 초대 생성\n                invitation = generate_calendar_invitation(\n                    state[\"recommendation\"],\n                    title=\"자동 조율 회의\"\n                )\n                state[\"invitation\"] = invitation\n                \n                # 알림 발송\n                payloads = generate_notification_payloads(\n                    invitation,\n                    channels=[\"teams\", \"email\"]\n                )\n                self.notification_sender.send_batch(payloads)\n                \n                state[\"status\"] = \"meeting_confirmed\"\n            \n            self.checkpoint_manager.save_checkpoint(\"node_8_confirm\", \n                                                    {\"status\": \"confirmed\"})\n            self.log_step(\"node_8_confirm\", \"success\", time.time() - start)\n            return state\n        except Exception as e:\n            self.log_step(\"node_8_confirm\", \"failed\", time.time() - start, str(e))\n            state[\"errors\"].append(f\"확정 실패: {str(e)}\")\n            return state\n    \n    def node_9_complete(self, state: dict) -> dict:\n        \"\"\"노드 9: 회의 일정 확정 완료\"\"\"\n        state[\"status\"] = \"workflow_complete\"\n        state[\"executed_steps\"].append(\"node_9_complete\")\n        return state\n\n\nprint(\"✓ LangGraph 워크플로우 구성 완료\")\nprint(f\"  - 총 9개 노드\")\nprint(f\"  - 1개 분기점 (node_6)\")

## Section 13: 노드 기반 워크플로우 오케스트레이션 (LangGraph)

In [ ]:
# 15. 일정 확정 및 알림 발송\nclass CalendarInvitation(BaseModel):\n    \"\"\"캘린더 초대\"\"\"\n    meeting_time: datetime\n    organizer: str\n    participants: List[str]\n    excluded_participants: List[str] = []\n    title: str\n    description: str\n    meeting_link: str\n    is_teams: bool = True\n\n\ndef generate_calendar_invitation(\n    recommendation: MeetingRecommendation,\n    organizer: str = \"HR\",\n    title: str = \"팀 전체 회의\"\n) -> CalendarInvitation:\n    \"\"\"캘린더 초대 생성\"\"\"\n    start_time = recommendation.recommended_time\n    end_time = start_time + timedelta(hours=1)  # 기본 1시간\n    \n    # Teams 회의 링크 생성 (시뮬레이션)\n    meeting_link = f\"https://teams.microsoft.com/l/meetup-join/{int(start_time.timestamp())}\"\n    \n    invitation = CalendarInvitation(\n        meeting_time=start_time,\n        organizer=organizer,\n        participants=recommendation.available_participants,\n        excluded_participants=recommendation.excluded_participants,\n        title=title,\n        description=f\"\"\"자동 조율된 회의입니다.\n신뢰도: {recommendation.confidence_score:.0%}\n사유: {recommendation.reason}\",\",\n        meeting_link=meeting_link,\n        is_teams=True\n    )\n    \n    return invitation\n\n\nclass NotificationPayload(BaseModel):\n    \"\"\"알림 페이로드\"\"\"\n    channel: str  # \"teams\", \"slack\", \"email\", \"messenger\"\n    recipient_id: str\n    title: str\n    message: str\n    action_url: str = \"\"\n    priority: str = \"normal\"  # \"high\", \"normal\", \"low\"\n    timestamp: datetime = Field(default_factory=lambda: datetime.now(KST))\n\n\ndef generate_notification_payloads(\n    invitation: CalendarInvitation,\n    channels: List[str] = [\"teams\", \"email\"]\n) -> List[NotificationPayload]:\n    \"\"\"다양한 채널의 알림 페이로드 생성\"\"\"\n    payloads = []\n    \n    meeting_time_str = invitation.meeting_time.strftime(\"%Y-%m-%d %H:%M\")\n    participants_str = \", \".join(invitation.participants)\n    \n    for channel in channels:\n        if channel == \"teams\":\n            payload = NotificationPayload(\n                channel=\"teams\",\n                recipient_id=\"team_channel\",\n                title=f\"🗓️ {invitation.title} 일정 확정\",\n                message=f\"\"\"회의가 자동으로 조율되었습니다!\n\n⏰ 시간: {meeting_time_str}\n👥 참석자: {participants_str}\n🔗 링크: {invitation.meeting_link}\n신뢰도: {recommendation.confidence_score:.0%}\",\",\n                action_url=invitation.meeting_link,\n                priority=\"high\"\n            )\n        \n        elif channel == \"email\":\n            payload = NotificationPayload(\n                channel=\"email\",\n                recipient_id=\"team@company.com\",\n                title=f\"[회의 초대] {invitation.title}\",\n                message=f\"\"\"안녕하세요,\n\n회의 일정이 자동으로 결정되었습니다.\n\n📅 날짜/시간: {meeting_time_str}\n👥 참석자: {participants_str}\n\n캘린더를 확인해주세요.\",\",\n                priority=\"normal\"\n            )\n        \n        elif channel == \"messenger\":\n            payload = NotificationPayload(\n                channel=\"messenger\",\n                recipient_id=\"internal_team_channel\",\n                title=\"회의 일정 안내\",\n                message=f\"{meeting_time_str} 회의 참석해주세요. {participants_str}\",\n                action_url=invitation.meeting_link,\n                priority=\"normal\"\n            )\n        \n        payloads.append(payload)\n    \n    return payloads\n\n\nclass MockNotificationSender:\n    \"\"\"알림 발송 시뮬레이터\"\"\"\n    def __init__(self):\n        self.sent_notifications = []\n    \n    def send(self, payload: NotificationPayload) -> dict:\n        \"\"\"알림 발송\"\"\"\n        self.sent_notifications.append(payload)\n        \n        status_emoji = \"✓\" if payload.priority == \"high\" else \"•\"\n        logger.info(\n            f\"{status_emoji} [{payload.channel.upper()}] {payload.recipient_id}: \"\n            f\"{payload.title}\"\n        )\n        \n        return {\n            \"status\": \"sent\",\n            \"channel\": payload.channel,\n            \"recipient\": payload.recipient_id,\n            \"timestamp\": datetime.now(KST).isoformat()\n        }\n    \n    def send_batch(self, payloads: List[NotificationPayload]) -> dict:\n        \"\"\"배치 발송\"\"\"\n        results = []\n        for payload in payloads:\n            result = self.send(payload)\n            results.append(result)\n        \n        return {\n            \"status\": \"completed\",\n            \"total_sent\": len(results),\n            \"results\": results\n        }\n\n\n# 테스트\nif alternatives:\n    print(\"✓ 알림 발송 및 캘린더 초대 자동화\")\n    \n    # 최고 점수 추천 선택\n    best_recommendation = alternatives[0]\n    \n    # 초대 생성\n    invitation = generate_calendar_invitation(\n        best_recommendation,\n        organizer=\"HR Manager\",\n        title=\"분기별 전략 수립 회의\"\n    )\n    \n    print(f\"\\n📅 캘린더 초대:\")\n    print(f\"  시간: {invitation.meeting_time.strftime('%Y-%m-%d %H:%M')}\")\n    print(f\"  참석자: {', '.join(invitation.participants)}\")\n    if invitation.excluded_participants:\n        print(f\"  불참: {', '.join(invitation.excluded_participants)}\")\n    print(f\"  링크: {invitation.meeting_link}\")\n    \n    # 알림 페이로드 생성\n    payloads = generate_notification_payloads(\n        invitation,\n        channels=[\"teams\", \"email\", \"messenger\"]\n    )\n    \n    print(f\"\\n📢 생성된 알림 페이로드: {len(payloads)}개\")\n    \n    # 알림 발송\n    sender = MockNotificationSender()\n    send_result = sender.send_batch(payloads)\n    \n    print(f\"\\n발송 결과:\")\n    print(f\"  상태: {send_result['status']}\")\n    print(f\"  총 발송: {send_result['total_sent']}개\")

## Section 12: 일정 확정, 알림 발송, 캘린더 초대 자동화

In [ ]:
# 14. 체크포인트 시스템 (상태 저장/복구)\nimport hashlib\nfrom pathlib import Path\n\nclass Checkpoint(BaseModel):\n    \"\"\"체크포인트 저장 단위\"\"\"\n    step_name: str\n    timestamp: datetime\n    version: int = 1\n    data: dict\n    checksum: str = \"\"\n    \n    def calculate_checksum(self) -> str:\n        \"\"\"데이터 무결성 검증용 체크섬\"\"\"\n        data_str = json.dumps(self.data, default=str, sort_keys=True)\n        return hashlib.md5(data_str.encode()).hexdigest()\n\n\nclass CheckpointManager:\n    \"\"\"체크포인트 관리자\"\"\"\n    def __init__(self, checkpoint_dir: str = \".checkpoints\"):\n        self.checkpoint_dir = Path(checkpoint_dir)\n        self.checkpoint_dir.mkdir(exist_ok=True)\n        self.checkpoints = {}  # 메모리 캐시\n    \n    def save_checkpoint(self, step_name: str, data: dict) -> Checkpoint:\n        \"\"\"체크포인트 저장\"\"\"\n        checkpoint = Checkpoint(\n            step_name=step_name,\n            timestamp=datetime.now(KST),\n            data=data\n        )\n        checkpoint.checksum = checkpoint.calculate_checksum()\n        \n        # 메모리에 저장\n        self.checkpoints[step_name] = checkpoint\n        \n        # 파일에 저장\n        checkpoint_file = self.checkpoint_dir / f\"{step_name}.json\"\n        with open(checkpoint_file, \"w\", encoding=\"utf-8\") as f:\n            json.dump(\n                {\n                    \"step_name\": checkpoint.step_name,\n                    \"timestamp\": checkpoint.timestamp.isoformat(),\n                    \"version\": checkpoint.version,\n                    \"checksum\": checkpoint.checksum,\n                    \"data\": checkpoint.data\n                },\n                f,\n                default=str,\n                ensure_ascii=False,\n                indent=2\n            )\n        \n        logger.info(f\"✓ 체크포인트 저장: {step_name} (checksum: {checkpoint.checksum[:8]}...)\")\n        return checkpoint\n    \n    def load_checkpoint(self, step_name: str) -> Optional[Checkpoint]:\n        \"\"\"체크포인트 로드\"\"\"\n        # 메모리 캐시 확인\n        if step_name in self.checkpoints:\n            return self.checkpoints[step_name]\n        \n        # 파일에서 로드\n        checkpoint_file = self.checkpoint_dir / f\"{step_name}.json\"\n        if checkpoint_file.exists():\n            with open(checkpoint_file, \"r\", encoding=\"utf-8\") as f:\n                data = json.load(f)\n            \n            checkpoint = Checkpoint(**data)\n            \n            # 체크섬 검증\n            expected = checkpoint.calculate_checksum()\n            if checkpoint.checksum != expected:\n                logger.warning(f\"⚠️ 체크포인트 무결성 오류: {step_name}\")\n                return None\n            \n            # 메모리에 캐시\n            self.checkpoints[step_name] = checkpoint\n            logger.info(f\"✓ 체크포인트 로드: {step_name}\")\n            return checkpoint\n        \n        return None\n    \n    def get_last_successful_step(self) -> Optional[str]:\n        \"\"\"마지막 성공한 단계 조회\"\"\"\n        if not self.checkpoints:\n            return None\n        \n        # 타임스탐프 기준 최신 것\n        latest = max(\n            self.checkpoints.values(),\n            key=lambda c: c.timestamp\n        )\n        return latest.step_name\n    \n    def list_checkpoints(self) -> list:\n        \"\"\"저장된 체크포인트 목록\"\"\"\n        checkpoints_list = []\n        for file in self.checkpoint_dir.glob(\"*.json\"):\n            step_name = file.stem\n            cp = self.load_checkpoint(step_name)\n            if cp:\n                checkpoints_list.append({\n                    \"step\": cp.step_name,\n                    \"timestamp\": cp.timestamp,\n                    \"version\": cp.version\n                })\n        return sorted(checkpoints_list, key=lambda x: x[\"timestamp\"])\n    \n    def clear_checkpoints(self):\n        \"\"\"모든 체크포인트 삭제\"\"\"\n        for file in self.checkpoint_dir.glob(\"*.json\"):\n            file.unlink()\n        self.checkpoints.clear()\n        logger.info(\"✓ 모든 체크포인트 삭제\")\n\n\n# 테스트\ncheckpoint_manager = CheckpointManager()\n\nprint(\"✓ 체크포인트 시스템 테스트\")\n\n# 단계별 저장\nsteps_data = {\n    \"step_1_input\": {\"participants\": [p.dict() for p in SAMPLE_PARTICIPANTS[:2]],\n                     \"duration_minutes\": 60},\n    \"step_2_collect\": {\"total_events\": 12, \"sources\": [\"Teams\", \"Internal DB\"]},\n    \"step_3_standardize\": {\"standardized_count\": 12, \"duplicates_removed\": 2},\n}\n\nfor step_name, data in steps_data.items():\n    checkpoint_manager.save_checkpoint(step_name, data)\n\nprint(\"\\n저장된 체크포인트:\")\nfor cp in checkpoint_manager.list_checkpoints():\n    print(f\"  - {cp['step']}: {cp['timestamp'].strftime('%H:%M:%S')}\")\n\nprint(f\"\\n마지막 성공 단계: {checkpoint_manager.get_last_successful_step()}\")\n\n# 복구 테스트\nrecovered = checkpoint_manager.load_checkpoint(\"step_2_collect\")\nif recovered:\n    print(f\"\\n복구된 데이터:\")\n    print(f\"  {recovered.data}\")

## Section 11: 상태 체크포인트 저장/복구 (DB/파일 기반)

In [ ]:
# 13. 조건부 재시도/폴백 전략
import time

class RetryConfig(BaseModel):
    \"\"\"재시도 설정\"\"\"\n    max_retries: int = 3\n    initial_delay_seconds: float = 0.1\n    backoff_factor: float = 2.0  # 지수 백오프\n    max_delay_seconds: float = 10.0\n\n\nclass CircuitBreaker:\n    \"\"\"회로차단기 패턴\"\"\"\n    def __init__(self, failure_threshold: int = 5, timeout_seconds: float = 60):\n        self.failure_threshold = failure_threshold\n        self.timeout_seconds = timeout_seconds\n        self.failure_count = 0\n        self.last_failure_time = None\n        self.state = \"CLOSED\"  # CLOSED, OPEN, HALF_OPEN\n    \n    def is_open(self) -> bool:\n        \"\"\"회로가 열려있는지 확인\"\"\"\n        if self.state == \"OPEN\":\n            if time.time() - self.last_failure_time > self.timeout_seconds:\n                self.state = \"HALF_OPEN\"\n                return False\n            return True\n        return False\n    \n    def record_success(self):\n        \"\"\"성공 기록\"\"\"\n        self.failure_count = 0\n        self.state = \"CLOSED\"\n    \n    def record_failure(self):\n        \"\"\"실패 기록\"\"\"\n        self.failure_count += 1\n        self.last_failure_time = time.time()\n        if self.failure_count >= self.failure_threshold:\n            self.state = \"OPEN\"\n\n\ndef retry_with_exponential_backoff(\n    func,\n    fallback_func,\n    config: RetryConfig = None,\n    circuit_breaker: CircuitBreaker = None,\n    *args,\n    **kwargs\n) -> dict:\n    \"\"\"\n    재시도 및 폴백 로직\n    1. 지수 백오프로 최대 N회 재시도\n    2. 모든 재시도 실패 시 폴백 실행\n    3. 회로차단기로 연속 실패 방지\n    \"\"\"\n    if config is None:\n        config = RetryConfig()\n    if circuit_breaker is None:\n        circuit_breaker = CircuitBreaker()\n    \n    # 회로가 열려있으면 바로 폴백\n    if circuit_breaker.is_open():\n        logger.warning(\"Circuit breaker OPEN: using fallback immediately\")\n        try:\n            result = fallback_func(*args, **kwargs)\n            result[\"fallback_reason\"] = \"circuit_breaker_open\"\n            return result\n        except Exception as e:\n            circuit_breaker.record_failure()\n            return {\"status\": \"failed\", \"error\": str(e), \"source\": \"fallback\"}\n    \n    delay = config.initial_delay_seconds\n    last_error = None\n    \n    # 재시도 루프\n    for attempt in range(config.max_retries):\n        try:\n            result = func(*args, **kwargs)\n            circuit_breaker.record_success()\n            result[\"attempt\"] = attempt + 1\n            result[\"status\"] = \"success\"\n            logger.info(f\"✓ 성공 (시도 {attempt + 1}/{config.max_retries})\")\n            return result\n        \n        except Exception as e:\n            last_error = e\n            circuit_breaker.record_failure()\n            logger.warning(f\"✗ 시도 {attempt + 1} 실패: {str(e)}\")\n            \n            if attempt < config.max_retries - 1:\n                logger.info(f\"  → {delay:.2f}초 후 재시도...\")\n                time.sleep(delay)\n                delay = min(delay * config.backoff_factor, config.max_delay_seconds)\n    \n    # 모든 재시도 실패 -> 폴백\n    logger.warning(f\"✗ 모든 재시도 실패. 폴백 함수 사용\")\n    try:\n        result = fallback_func(*args, **kwargs)\n        result[\"fallback_reason\"] = \"all_retries_failed\"\n        result[\"original_error\"] = str(last_error)\n        circuit_breaker.record_failure()  # 폴백도 기록\n        return result\n    except Exception as fallback_error:\n        circuit_breaker.record_failure()\n        return {\n            \"status\": \"failed\",\n            \"error\": str(last_error),\n            \"fallback_error\": str(fallback_error),\n            \"source\": \"both_failed\"\n        }\n\n\n# 테스트: 실패 가능한 함수\nfailure_count = 0\n\ndef unstable_api_call(participant_name: str) -> dict:\n    \"\"\"불안정한 API (처음 2회는 실패)\"\"\"\n    global failure_count\n    failure_count += 1\n    \n    if failure_count <= 2:\n        raise Exception(f\"API 일시적 오류 (시도 {failure_count})\")\n    \n    return {\n        \"status\": \"success\",\n        \"data\": f\"{participant_name} 일정 데이터\"\n    }\n\n\ndef fallback_api_call(participant_name: str) -> dict:\n    \"\"\"폴백: 캐시된 데이터 사용\"\"\"\n    return {\n        \"status\": \"fallback\",\n        \"data\": f\"{participant_name} 캐시된 일정 (최근 24시간)\"\n    }\n\n\n# 테스트 실행\nprint(\"✓ 재시도/폴백 전략 테스트\")\nprint(\"\\n[테스트 1] 재시도 후 성공\")\nfailure_count = 0\nresult = retry_with_exponential_backoff(\n    unstable_api_call,\n    fallback_api_call,\n    participant_name=\"Alice\"\n)\nprint(f\"결과: {result}\\n\")\n\n# 테스트 2: 완전 실패 -> 폴백\ndef always_fail(participant_name: str) -> dict:\n    raise Exception(\"영구적 API 오류\")\n\nprint(\"[테스트 2] 완전 실패 후 폴백\")\nresult = retry_with_exponential_backoff(\n    always_fail,\n    fallback_api_call,\n    config=RetryConfig(max_retries=2),\n    participant_name=\"Bob\"\n)\nprint(f\"결과: {result}\")

## Section 10: 조건부 재시도/폴백 전략 구현

In [ ]:
# 12. 최적 회의 시간 스코어링
class ScoredRecommendation(BaseModel):
    """점수가 매겨진 추천"""
    recommendation: MeetingRecommendation
    base_score: float  # 신뢰도 (0-1)
    time_preference_score: float  # 시간 선호도 (0-1)
    priority_score: float  # 우선순위 반영 (0-1)
    total_score: float  # 최종 점수


def calculate_time_preference_score(meeting_time: datetime) -> float:
    \"\"\"\n    시간대 선호도 점수\n    - 오전 10-11시: 1.0 (최고)\n    - 오후 14-17시: 0.8\n    - 기타: 0.6\n    \"\"\"\n    hour = meeting_time.hour\n    \n    if 10 <= hour < 12:\n        return 1.0\n    elif 14 <= hour < 17:\n        return 0.8\n    elif 9 <= hour < 18:\n        return 0.6\n    else:\n        return 0.2  # 업무 시간 외\n\n\ndef calculate_priority_score(\n    recommendation: MeetingRecommendation,\n    all_participants: List[Participant]\n) -> float:\n    \"\"\"\n    참가자 우선순위 점수\n    - HIGH priority가 모두 참석하면 1.0\n    - MEDIUM priority도 포함되면 0.8\n    \"\"\"\n    available_set = set(recommendation.available_participants)\n    high_priority = [p for p in all_participants if p.priority == Priority.HIGH]\n    medium_priority = [p for p in all_participants if p.priority == Priority.MEDIUM]\n    \n    high_available = sum(1 for p in high_priority if p.name in available_set)\n    medium_available = sum(1 for p in medium_priority if p.name in available_set)\n    \n    high_ratio = high_available / max(1, len(high_priority))\n    medium_ratio = medium_available / max(1, len(medium_priority))\n    \n    return high_ratio * 0.7 + medium_ratio * 0.3\n\n\ndef score_recommendations(\n    recommendations: List[MeetingRecommendation],\n    participants: List[Participant],\n    weights: dict = None\n) -> List[ScoredRecommendation]:\n    \"\"\"\n    추천을 점수화하고 랭킹\n    \n    스코어 공식:\n    total_score = w1*base + w2*time + w3*priority\n    \"\"\"\n    if weights is None:\n        weights = {\"base\": 0.4, \"time\": 0.3, \"priority\": 0.3}\n    \n    scored = []\n    for rec in recommendations:\n        base_score = rec.confidence_score\n        time_score = calculate_time_preference_score(rec.recommended_time)\n        priority_score = calculate_priority_score(rec, participants)\n        \n        total_score = (\n            weights[\"base\"] * base_score +\n            weights[\"time\"] * time_score +\n            weights[\"priority\"] * priority_score\n        )\n        \n        scored.append(ScoredRecommendation(\n            recommendation=rec,\n            base_score=base_score,\n            time_preference_score=time_score,\n            priority_score=priority_score,\n            total_score=total_score\n        ))\n    \n    # 점수 내림차순 정렬\n    scored.sort(key=lambda s: s.total_score, reverse=True)\n    return scored\n\n\n# 스코어링 실행\nif alternatives:\n    scored_recs = score_recommendations(alternatives, SAMPLE_PARTICIPANTS)\n    \n    print(\"✓ 최적 회의 시간 스코어링 완료\")\n    print(\"\\n점수 계산 결과 (상위 3개):\")\n    print(f\"{'순위':<3} {'시간':<20} {'신뢰도':<8} {'시간선호':<8} {'우선순위':<8} {'최종점수':<8}\")\n    print(\"-\" * 60)\n    \n    for i, scored in enumerate(scored_recs[:3], 1):\n        print(\n            f\"{i:<3} \"\n            f\"{scored.recommendation.recommended_time.strftime('%Y-%m-%d %H:%M'):<20} \"\n            f\"{scored.base_score:<8.2f} \"\n            f\"{scored.time_preference_score:<8.2f} \"\n            f\"{scored.priority_score:<8.2f} \"\n            f\"{scored.total_score:<8.3f}\"\n        )\n        print(f\"     → {', '.join(scored.recommendation.available_participants)}\")

## Section 9: 최적 회의 시간 스코어링 및 랭킹

In [ ]:
# 11. 대체안 생성 (공통 시간이 없을 때)
def generate_alternative_recommendations(
    available_slots_by_participant: Dict[str, List[TimeSlot]],
    participants: List[Participant],
    required_count: int = None
) -> List[MeetingRecommendation]:
    """
    공통 시간이 없을 경우 대체안 생성
    - 필수 참석자는 반드시 포함
    - 우선순위 높은 사람부터 포함
    """
    if required_count is None:
        required_count = max(1, len(participants) - 1)  # 1명 빼도 됨
    
    # 우선순위 정렬
    sorted_participants = sorted(
        participants,
        key=lambda p: (p.priority == Priority.HIGH, p.priority == Priority.MEDIUM),
        reverse=True
    )
    
    # 각 참가자 조합별로 가능한 시간 찾기
    all_slots_by_time = defaultdict(set)
    for participant_name, slots in available_slots_by_participant.items():
        for slot in slots:
            key = (slot.start_time, slot.end_time)
            all_slots_by_time[key].add(participant_name)
    
    recommendations = []
    
    for (start_time, end_time), available_set in all_slots_by_time.items():
        available_count = len(available_set)
        
        if available_count >= required_count:
            # HIGH priority 참가자들을 우선적으로 포함
            high_priority = [p for p in sorted_participants if p.priority == Priority.HIGH]
            medium_priority = [p for p in sorted_participants if p.priority == Priority.MEDIUM]
            
            selected = []
            excluded = []
            
            # HIGH priority 우선 포함
            for p in high_priority:
                if p.name in available_set:
                    selected.append(p.name)
                else:
                    excluded.append(p.name)
            
            # 불족한 경우 MEDIUM priority 추가
            if len(selected) < required_count:
                for p in medium_priority:
                    if len(selected) >= required_count:
                        break
                    if p.name in available_set:
                        selected.append(p.name)
                    else:
                        excluded.append(p.name)
            
            if len(selected) >= required_count:
                confidence = available_count / len(participants)
                recommendations.append(MeetingRecommendation(
                    recommended_time=start_time,
                    available_participants=selected,
                    excluded_participants=excluded,
                    confidence_score=confidence,
                    reason=f"{len(selected)}명 가능 (제외: {', '.join(excluded) if excluded else '없음'})"
                ))
    
    # 신뢰도 점수 + 시간순으로 정렬
    recommendations.sort(
        key=lambda r: (-r.confidence_score, r.recommended_time)
    )
    
    return recommendations[:10]  # 상위 10개만 반환


# 대체안 생성
alternatives = generate_alternative_recommendations(
    available_slots_by_participant,
    SAMPLE_PARTICIPANTS
)

print(f"✓ 대체안 생성 완료")
print(f"  - 추천 시간대: {len(alternatives)}개")

if alternatives:
    print("\n추천 대체 시간대 (상위 3개):")
    for i, rec in enumerate(alternatives[:3], 1):
        print(f"\n{i}. {rec.recommended_time.strftime('%Y-%m-%d %H:%M')}")
        print(f"   참석: {', '.join(rec.available_participants)} ({len(rec.available_participants)}명)")
        if rec.excluded_participants:
            print(f"   제외: {', '.join(rec.excluded_participants)}")
        print(f"   신뢰도: {rec.confidence_score:.2%} | {rec.reason}")

## Section 8: 공통 시간 부재 시 대체안 생성 (우선순위/제외 인원 반영)

In [ ]:
# 10. 전체 참가자 교집합 시간대 탐색
def find_common_slots(
    available_slots_by_participant: Dict[str, List[TimeSlot]],
    participants: List[Participant]
) -> List[TimeSlot]:
    """
    모든 참가자가 가능한 시간대 찾기 (교집합)
    """
    if not available_slots_by_participant:
        return []
    
    # 모든 슬롯을 시간대로 매핑
    all_slots_by_time = defaultdict(list)
    
    for participant_name, slots in available_slots_by_participant.items():
        for slot in slots:
            key = (slot.start_time, slot.end_time)
            if participant_name not in all_slots_by_time[key]:
                all_slots_by_time[key].append(participant_name)
    
    # 모든 참가자가 가능한 시간대만 필터링
    all_participant_names = {p.name for p in participants}
    common_slots = []
    
    for (start_time, end_time), participant_list in all_slots_by_time.items():
        if set(participant_list) == all_participant_names:
            common_slots.append(TimeSlot(
                start_time=start_time,
                end_time=end_time,
                available_participants=list(participant_list)
            ))
    
    # 시간순 정렬
    common_slots.sort(key=lambda s: s.start_time)
    return common_slots


# 교집합 계산
common_slots = find_common_slots(available_slots_by_participant, SAMPLE_PARTICIPANTS)
print(f"✓ 교집합 시간대 탐색 완료")
print(f"  - 모든 참가자가 가능한 시간대: {len(common_slots)}개")

if common_slots:
    print("\n공통 가능 시간대 (상위 5개):")
    for slot in common_slots[:5]:
        print(f"  ✓ {slot.start_time.strftime('%Y-%m-%d %H:%M')} ~ {slot.end_time.strftime('%H:%M')}")
        print(f"     참가자: {', '.join(slot.available_participants)}")
else:
    print("\n⚠️ 모든 참가자가 함께 가능한 시간대가 없습니다!")

# 부분 교집합 (일부 참가자만 필요한 경우)
def find_partial_slots(
    available_slots_by_participant: Dict[str, List[TimeSlot]],
    min_participants_count: int
) -> List[TimeSlot]:
    """
    최소 N명 이상이 가능한 시간대 찾기
    """
    all_slots_by_time = defaultdict(set)
    
    for participant_name, slots in available_slots_by_participant.items():
        for slot in slots:
            key = (slot.start_time, slot.end_time)
            all_slots_by_time[key].add(participant_name)
    
    partial_slots = []
    for (start_time, end_time), participant_set in all_slots_by_time.items():
        if len(participant_set) >= min_participants_count:
            partial_slots.append(TimeSlot(
                start_time=start_time,
                end_time=end_time,
                available_participants=list(participant_set)
            ))
    
    partial_slots.sort(key=lambda s: s.start_time)
    return partial_slots


# 부분 교집합 계산
for min_count in [5, 4, 3]:
    partial = find_partial_slots(available_slots_by_participant, min_count)
    if partial:
        print(f"\n{min_count}명 이상 가능한 시간: {len(partial)}개")
        print(f"  샘플: {partial[0].start_time.strftime('%Y-%m-%d %H:%M')} ~ {', '.join(partial[0].available_participants)}")
        break

## Section 7: 전체 참가자 교집합 시간대 탐색

In [ ]:
# 9. 개인별 가용 시간대 계산
WORK_START_HOUR = 9
WORK_END_HOUR = 18
LUNCH_START = 12
LUNCH_END = 13
SLOT_DURATION_MINUTES = 30


def calculate_available_slots(
    participant_name: str,
    events: List[CalendarEvent],
    date_range: Tuple[datetime, datetime],
    duration_minutes: int = 60
) -> List[TimeSlot]:
    """
    개인별 가용 시간대 계산 (30분 단위 슬롯)
    
    Args:
        participant_name: 참가자 이름
        events: 해당 참가자의 일정 목록
        date_range: 검색 날짜 범위 (start, end)
        duration_minutes: 필요 회의 시간
    
    Returns:
        TimeSlot 리스트
    """
    available_slots = []
    
    # 업무 시간 범위으로 시간대 생성
    current_date = date_range[0].date()
    end_date = date_range[1].date()
    
    while current_date <= end_date:
        # 점심 시간 제외한 업무 시간대
        day_start = datetime.combine(current_date, time(WORK_START_HOUR, 0), tzinfo=KST)
        day_end = datetime.combine(current_date, time(WORK_END_HOUR, 0), tzinfo=KST)
        
        current_time = day_start
        
        while current_time + timedelta(minutes=duration_minutes) <= day_end:
            slot_end = current_time + timedelta(minutes=duration_minutes)
            
            # 점심 시간 제외
            if LUNCH_START <= current_time.hour < LUNCH_END or LUNCH_START <= slot_end.hour < LUNCH_END:
                current_time += timedelta(minutes=SLOT_DURATION_MINUTES)
                continue
            
            # 기존 일정과 겹치는지 확인
            has_conflict = False
            for event in events:
                if event.start_time < slot_end and event.end_time > current_time:
                    has_conflict = True
                    break
            
            if not has_conflict:
                available_slots.append(TimeSlot(
                    start_time=current_time,
                    end_time=slot_end,
                    available_participants=[participant_name]
                ))
            
            current_time += timedelta(minutes=SLOT_DURATION_MINUTES)
        
        current_date += timedelta(days=1)
    
    return available_slots


# 각 참가자의 가용 시간대 계산
today = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0, tzinfo=KST)
date_range = (today, today + timedelta(days=4))  # 5일간

available_slots_by_participant = {}
for participant_name, events in normalized_calendar.items():
    slots = calculate_available_slots(participant_name, events, date_range)
    available_slots_by_participant[participant_name] = slots

print("✓ 개인별 가용 시간대 계산 완료")
print(f"  - 검색 기간: {date_range[0].date()} ~ {date_range[1].date()}")
for name, slots in available_slots_by_participant.items():
    print(f"    {name}: {len(slots):3}개 슬롯 ({len(slots) * SLOT_DURATION_MINUTES}분)")

# 가용 시간대 시각화
print("\n개인별 가용 시간대 (샘플 - 첫 2일):")
for participant_name, slots in available_slots_by_participant.items():
    print(f"\n[{participant_name}]")
    for slot in slots[:4]:
        print(f"  {slot.start_time.strftime('%Y-%m-%d %H:%M')} ~ {slot.end_time.strftime('%H:%M')}")

## Section 6: 개인별 가용 시간대 계산 엔진

In [ ]:
# 8. 불변 일정 식별 (LLM 기반 또는 규칙 기반)
# 키워드 기반 불변 일정 식별
FIXED_KEYWORDS = [
    "고객", "customer", "client", "external",
    "고정", "fixed", "immutable",
    "중요", "critical", "essential",
    "데드라인", "deadline",
]

def identify_fixed_schedules_rule_based(
    calendar_data: Dict[str, List[CalendarEvent]]
) -> List[CalendarEvent]:
    """
    규칙 기반 불변 일정 식별
    키워드 매칭 또는 is_fixed 플래그 사용
    """
    fixed_schedules = []
    
    for participant_name, events in calendar_data.items():
        for event in events:
            # 이미 is_fixed 플래그가 설정된 경우
            if event.is_fixed or event.is_external:
                fixed_schedules.append(event)
            else:
                # 제목에서 키워드 확인
                title_lower = event.title.lower()
                for keyword in FIXED_KEYWORDS:
                    if keyword.lower() in title_lower:
                        event.is_fixed = True
                        fixed_schedules.append(event)
                        break
    
    return fixed_schedules


# 불변 일정 식별
fixed_schedules = identify_fixed_schedules_rule_based(normalized_calendar)
print("✓ 불변 일정 식별 완료")
print(f"  - 총 불변 일정: {len(fixed_schedules)}건")
for event in fixed_schedules:
    print(f"    [{event.participant}] {event.title} ({event.start_time.strftime('%Y-%m-%d %H:%M')})")


# 신뢰도 점수 계산
class FixedScheduleScore(BaseModel):
    """불변 일정 신뢰도 점수"""
    event: CalendarEvent
    confidence_score: float
    reason: str


def calculate_fixed_schedule_confidence(
    calendar_data: Dict[str, List[CalendarEvent]]
) -> List[FixedScheduleScore]:
    """불변 일정 신뢰도 점수 계산"""
    scores = []
    
    for participant_name, events in calendar_data.items():
        for event in events:
            confidence = 0.0
            reason_parts = []
            
            # 기준 1: is_fixed 플래그 (100점)
            if event.is_fixed:
                confidence += 0.5
                reason_parts.append("is_fixed=True")
            
            # 기준 2: is_external (40점)
            if event.is_external:
                confidence += 0.4
                reason_parts.append("외부 일정")
            
            # 기준 3: 키워드 매칭
            title_lower = event.title.lower()
            for keyword in FIXED_KEYWORDS:
                if keyword.lower() in title_lower:
                    confidence += 0.1
                    reason_parts.append(f"키워드: {keyword}")
                    break
            
            # 기준 4: 참가자 우선순위
            if participant_name in ["Alice", "Eve"]:  # HIGH priority
                confidence += 0.05
                reason_parts.append("HIGH priority participant")
            
            scores.append(FixedScheduleScore(
                event=event,
                confidence_score=min(confidence, 1.0),
                reason=" | ".join(reason_parts) if reason_parts else "일반 일정"
            ))
    
    return scores


# 신뢰도 점수 계산
confidence_scores = calculate_fixed_schedule_confidence(normalized_calendar)
print("\n불변 일정 신뢰도 점수:")
for score in confidence_scores:
    if score.confidence_score > 0.5:
        print(f"  [{score.event.participant:10}] {score.event.title:30} | 신뢰도: {score.confidence_score:.2f} | {score.reason}")

## Section 5: 불변 일정(고객사 등) 태깅 모델/규칙 구현

In [ ]:
# 7. 일정 표준화 및 정규화
def standardize_calendar_data(
    calendar_data: Dict[str, List[CalendarEvent]]
) -> Dict[str, List[CalendarEvent]]:
    """
    원시 날짜 표준화, 중복 제거, 결측치 보정
    """
    standardized = {}
    
    for participant_name, events in calendar_data.items():
        # 시간순 정렬
        sorted_events = sorted(events, key=lambda e: e.start_time)
        
        # 중복 제거 (같은 시간대, 같은 제목)
        unique_events = []
        seen = set()
        
        for event in sorted_events:
            key = (event.start_time, event.end_time, event.title, event.participant)
            if key not in seen:
                seen.add(key)
                unique_events.append(event)
        
        # 겹치는 이벤트 병합
        merged_events = []
        for event in unique_events:
            if merged_events and merged_events[-1].end_time >= event.start_time:
                # 이전 이벤트와 겹침 -> 병합
                last = merged_events[-1]
                merged_event = CalendarEvent(
                    title=f"{last.title} + {event.title}",
                    start_time=last.start_time,
                    end_time=max(last.end_time, event.end_time),
                    is_fixed=last.is_fixed or event.is_fixed,
                    is_external=last.is_external or event.is_external,
                    participant=participant_name
                )
                merged_events[-1] = merged_event
            else:
                merged_events.append(event)
        
        standardized[participant_name] = merged_events
    
    return standardized


# 표준화 실행
standardized_calendar = standardize_calendar_data(calendar_sample)
print("✓ 일정 표준화 완료")
print(f"  - 표준화 후 일정 수")
for name, events in standardized_calendar.items():
    print(f"    {name}: {len(events)}건 (변경 불가: {sum(1 for e in events if e.is_fixed)}건)")


# 타임존 정규화 (한국 표준시)
from zoneinfo import ZoneInfo

KST = ZoneInfo("Asia/Seoul")

def normalize_timezone(calendar_data: Dict[str, List[CalendarEvent]]) -> Dict[str, List[CalendarEvent]]:
    """모든 시간을 KST로 정규화"""
    normalized = {}
    
    for participant_name, events in calendar_data.items():
        normalized_events = []
        for event in events:
            # naive datetime을 KST로 인식
            if event.start_time.tzinfo is None:
                start = event.start_time.replace(tzinfo=KST)
            else:
                start = event.start_time.astimezone(KST)
            
            if event.end_time.tzinfo is None:
                end = event.end_time.replace(tzinfo=KST)
            else:
                end = event.end_time.astimezone(KST)
            
            normalized_event = CalendarEvent(
                title=event.title,
                start_time=start,
                end_time=end,
                is_fixed=event.is_fixed,
                is_external=event.is_external,
                participant=participant_name
            )
            normalized_events.append(normalized_event)
        
        normalized[participant_name] = normalized_events
    
    return normalized


# 타임존 정규화 실행
normalized_calendar = normalize_timezone(standardized_calendar)
print("\n✓ 타임존 정규화 완료 (KST)")

# 최종 데이터 확인
final_df = calendar_to_dataframe(normalized_calendar)
print("\n최종 캘린더 데이터:")
print(final_df.to_string(index=False))

## Section 4: 일정 표준화 및 타임존 정규화

In [ ]:
# 6. 다중 소스 데이터 수집 파이프라인
def fetch_calendar_from_multiple_sources(
    participant: Participant,
    connectors_list: List[CalendarConnector],
    simulate_failure: bool = False,
    failure_rate: float = 0.0
) -> List[CalendarEvent]:
    """
    여러 소스에서 참가자 일정 수집
    
    Args:
        participant: 대상 참가자
        connectors_list: 사용할 커넥터 리스트
        simulate_failure: 실패 시뮬레이션 여부
        failure_rate: 실패율 (0-1)
    """
    all_events = []
    import random
    
    for connector in connectors_list:
        try:
            # 실패 시뮬레이션
            if simulate_failure and random.random() < failure_rate:
                raise Exception(f"{connector.name} API 오류")
            
            events = connector.fetch_calendar(participant)
            all_events.extend(events)
        except Exception as e:
            logger.warning(f"{connector.name}에서 {participant.name} 일정 수집 실패: {str(e)}")
            continue
    
    return all_events


def collect_all_participants_calendar(
    participants: List[Participant],
    simulate_failure: bool = False
) -> Dict[str, List[CalendarEvent]]:
    """모든 참가자의 일정 수집"""
    calendar_data = {}
    connectors_list = list(connectors.values())
    
    for participant in participants:
        events = fetch_calendar_from_multiple_sources(
            participant,
            connectors_list,
            simulate_failure=simulate_failure
        )
        calendar_data[participant.name] = events
    
    return calendar_data


# 테스트: 정상 수집
calendar_sample = collect_all_participants_calendar(SAMPLE_PARTICIPANTS)
print(f"✓ 캘린더 데이터 수집 완료")
print(f"  - 참가자별 일정 수")
for name, events in calendar_sample.items():
    print(f"    {name}: {len(events)}건")


# 테스트: 실패 시뮬레이션
print("\n⚠️ 실패 시뮬레이션 테스트")
calendar_with_failure = collect_all_participants_calendar(
    SAMPLE_PARTICIPANTS,
    simulate_failure=True
)
print(f"✓ 일부 실패 후에도 수집 완료")

# DataFrame으로 병합
def calendar_to_dataframe(calendar_data: Dict[str, List[CalendarEvent]]) -> pd.DataFrame:
    """캘린더 데이터를 DataFrame으로 변환"""
    rows = []
    for participant_name, events in calendar_data.items():
        for event in events:
            rows.append({
                'participant': participant_name,
                'title': event.title,
                'start_time': event.start_time,
                'end_time': event.end_time,
                'is_fixed': event.is_fixed,
                'is_external': event.is_external,
            })
    return pd.DataFrame(rows)

calendar_df = calendar_to_dataframe(calendar_sample)
print("\n캘린더 데이터 (DataFrame):")
print(calendar_df.head(10))

## Section 3: 사내 시스템 데이터 수집 파이프라인

In [ ]:
# 3. 샘플 참가자 데이터
SAMPLE_PARTICIPANTS = [
    Participant(name="Alice", email="alice@company.com", department="Sales", priority=Priority.HIGH),
    Participant(name="Bob", email="bob@company.com", department="Engineering", priority=Priority.HIGH),
    Participant(name="Charlie", email="charlie@company.com", department="Marketing", priority=Priority.MEDIUM),
    Participant(name="Diana", email="diana@company.com", department="Operations", priority=Priority.MEDIUM),
    Participant(name="Eve", email="eve@company.com", department="Executive", priority=Priority.HIGH),
]

# 4. 더미 캘린더 데이터 생성
def generate_sample_calendar_data() -> Dict[str, List[CalendarEvent]]:
    """테스트용 샘플 캘린더 데이터"""
    today = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
    base_date = today + timedelta(days=1)  # 내일부터
    
    calendar_data = {
        "Alice": [
            CalendarEvent(
                title="고객 미팅 (고정)",
                start_time=base_date + timedelta(hours=10),
                end_time=base_date + timedelta(hours=11),
                is_fixed=True,
                is_external=True,
                participant="Alice"
            ),
            CalendarEvent(
                title="팀 스탠드업",
                start_time=base_date + timedelta(hours=9),
                end_time=base_date + timedelta(hours=9, minutes=30),
                participant="Alice"
            ),
        ],
        "Bob": [
            CalendarEvent(
                title="프로젝트 리뷰",
                start_time=base_date + timedelta(hours=14),
                end_time=base_date + timedelta(hours=15),
                participant="Bob"
            ),
        ],
        "Charlie": [
            CalendarEvent(
                title="마케팅 회의",
                start_time=base_date + timedelta(hours=15),
                end_time=base_date + timedelta(hours=16, minutes=30),
                participant="Charlie"
            ),
        ],
        "Diana": [
            CalendarEvent(
                title="운영 회의",
                start_time=base_date + timedelta(hours=11),
                end_time=base_date + timedelta(hours=12),
                participant="Diana"
            ),
        ],
        "Eve": [
            CalendarEvent(
                title="경영진 회의",
                start_time=base_date + timedelta(hours=9),
                end_time=base_date + timedelta(hours=10),
                is_fixed=True,
                participant="Eve"
            ),
        ],
    }
    return calendar_data

# 5. 커넥터 인터페이스
from abc import ABC, abstractmethod

class CalendarConnector(ABC):
    """캘린더 데이터 수집 커넥터 추상 클래스"""
    
    def __init__(self, name: str):
        self.name = name
    
    @abstractmethod
    def fetch_calendar(self, participant: Participant) -> List[CalendarEvent]:
        """참가자의 캘린더 데이터 수집"""
        pass
    
    def __repr__(self):
        return f"{self.name}Connector"


class TeamsConnector(CalendarConnector):
    """Teams API 커넥터"""
    def fetch_calendar(self, participant: Participant) -> List[CalendarEvent]:
        logger.info(f"[Teams] 수집 중: {participant.name}")
        sample_data = generate_sample_calendar_data()
        return sample_data.get(participant.name, [])


class InternalDBConnector(CalendarConnector):
    """사내 DB 커넥터"""
    def fetch_calendar(self, participant: Participant) -> List[CalendarEvent]:
        logger.info(f"[Internal DB] 수집 중: {participant.name}")
        sample_data = generate_sample_calendar_data()
        return sample_data.get(participant.name, [])


class MessengerConnector(CalendarConnector):
    """메신저 연동 커넥터"""
    def fetch_calendar(self, participant: Participant) -> List[CalendarEvent]:
        logger.info(f"[Messenger] 수집 중: {participant.name}")
        return []  # 메신저에서는 일정 없음

# 커넥터 인스턴스
connectors = {
    "teams": TeamsConnector("Teams"),
    "internal_db": InternalDBConnector("Internal DB"),
    "messenger": MessengerConnector("Messenger"),
}

print("✓ 샘플 데이터 및 커넥터 구현 완료")
print(f"  - 참가자: {len(SAMPLE_PARTICIPANTS)}명")
print(f"  - 커넥터: {', '.join(connectors.keys())}")

## Section 2: 참가자/일정 더미 데이터 및 커넥터 인터페이스

In [ ]:
# 2. 데이터 스키마 정의
class Priority(str, Enum):
    """참가자 우선순위"""
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"


class Participant(BaseModel):
    """회의 참가자 정보"""
    name: str = Field(..., description="참가자 이름")
    email: str = Field(..., description="참가자 이메일")
    department: str = Field(..., description="부서")
    priority: Priority = Field(default=Priority.MEDIUM, description="우선순위")


class CalendarEvent(BaseModel):
    """캘린더 이벤트"""
    title: str
    start_time: datetime
    end_time: datetime
    is_fixed: bool = False  # 변경 불가능 여부
    is_external: bool = False  # 외부 일정 여부
    participant: str


class TimeSlot(BaseModel):
    """가용 시간대"""
    start_time: datetime
    end_time: datetime
    available_participants: List[str]


class MeetingRecommendation(BaseModel):
    """회의 추천"""
    recommended_time: datetime
    available_participants: List[str]
    excluded_participants: List[str] = []
    confidence_score: float
    reason: str


class AgentState(BaseModel):
    """워크플로우 상태"""
    step: str = "input"
    participants: List[Participant] = []
    required_duration_minutes: int = 60
    preferred_date_range: Optional[Tuple[datetime, datetime]] = None
    calendar_data: Dict[str, List[CalendarEvent]] = {}
    fixed_schedules: List[CalendarEvent] = []
    common_available_slots: List[TimeSlot] = []
    has_common_slot: bool = False
    final_recommendation: Optional[MeetingRecommendation] = None
    checkpoint_data: Dict = {}
    error_log: List[str] = []
    retry_count: int = 0


print("✓ Pydantic 스키마 정의 완료")

In [ ]:
# 1. 필수 라이브러리 임포트
import sys
sys.path.append('.')

import os
from dotenv import load_dotenv
load_dotenv()

from datetime import datetime, timedelta, time
from typing import Optional, Dict, List, Tuple
from pydantic import BaseModel, Field
from enum import Enum
import json
import pandas as pd
from collections import defaultdict
import logging
from functools import wraps

# LangGraph 관련
from langgraph.graph import StateGraph, START, END
from langgraph.types import StateSnapshot

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ 라이브러리 임포트 완료")
print(f"  - pandas: {pd.__version__}")
print("  - LangGraph: 설치됨")

## Section 1: 환경 설정 및 데이터 스키마 정의

필요한 라이브러리 임포트 및 Pydantic 스키마 정의

# MeetingScheduler: 회의 일정 자동 조율 Agent

## 개요
이 노트북은 **회의 참가자들의 일정을 자동으로 수집·분석하여 최적의 회의 시간을 추천하고 확정**하는 LangGraph 기반 Agent입니다.

### 주요 기능
- 👥 **참가자 일정 통합 수집**: Teams, 사내 시스템, 메신저에서 자동 수집
- 📌 **불변 일정 식별**: 고객사 미팅 등 변경 불가 일정 태깅
- ⏰ **교집합 시간대 탐색**: 모든 참가자가 가능한 시간 찾기
- 🔄 **대체 시간 추천**: 공통 시간 없을 때 우선순위 반영
- 📧 **자동 알림**: Teams/메신저를 통한 초대장 발송
- 🔁 **폴백 및 재시도**: API 실패 시 자동 복구
- 💾 **상태 체크포인트**: 단계별 결과 저장 및 복구

### 워크플로우
```
입력 → 수집 → 정리 → 태깅 → 교집합 판단 → 대체안/확정 → 알림 → 완료
```

### 고급 요소
1. **조건부 재시도/폴백**: API 실패 시 임계값 재시도 후 규칙 기반 대체
2. **상태 체크포인트**: 실패 시 마지막 성공 지점부터 재개
3. **최적 회의 시간 추출**: 우선순위 + 중요도 + 선호도 가중합
